<a target="_blank" href="https://colab.research.google.com/github/AshishKumar4/dew/blob/main/tutorials/02-train-a-diffusion-model.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Train a diffusion model with Dew

In this notebook you train a real diffusion model on real photographs and generate samples from it. The model is a DiT, a transformer that sees the image as a grid of patches and predicts the noise on each patch. The dataset is Oxford Flowers, 8189 photographs of 102 species. Training uses the EDM formulation of Karras et al. (2022), which is what the current crop of open image models builds on, and sampling uses the Euler ancestral solver.

This is the first notebook that uses the library itself rather than building diffusion by hand. [Notebook 01](01-diffusion-from-scratch.ipynb) built the theory; here every piece you wrote there comes from Dew, and the trainer takes over the parts that are the same for every model: the device mesh, the compiled step, the exponential moving average of the weights, checkpoints and logging.

**Expected time**: about 45 minutes on a Colab A100, about 70 minutes on a Colab T4 or a TPU v5e, about 55 minutes on a local RTX 4080. The first run also downloads the dataset, about 330 MB. Lower `EPOCHS` in the configuration cell to trade quality for time; 20 epochs already shows recognizable flower shapes.

In [ ]:
# On Colab: install dew and the JAX build for the runtime. Locally this cell is a no-op.
try:
    import google.colab  # noqa: F401
    import subprocess, sys
    try:
        import jax
        tpu = any("tpu" in str(d).lower() for d in jax.devices())
    except Exception:
        tpu = False
    extra = "jax[tpu]" if tpu else "jax[cuda12]"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "dew-ml[tfds] @ git+https://github.com/AshishKumar4/dew", extra, "matplotlib"])
except ImportError:
    pass

In [ ]:
# The whole run in one place. Everything below derives from these numbers.
BATCH_SIZE = 64           # images per optimizer step
IMAGE_SIZE = 64           # pixels; 128 needs roughly four times the compute
EPOCHS = 200              # passes over the dataset
LEARNING_RATE = 2e-4
MODEL = dict(patch_size=4, emb_features=384, num_layers=8, num_heads=6)
RUN_NAME = "flowers-dit"  # checkpoints land in ./checkpoints/<RUN_NAME>
WANDB_PROJECT = None      # set to a project name to log the run, see the last section
WORKER_COUNT = 4          # grain worker processes for the data pipeline
SAMPLES = 16              # images in the final grid
SAMPLE_STEPS = 40         # solver steps at sampling time
SEED = 0

In [ ]:
import jax

print(jax.devices())
print(jax.default_backend())

## The data

Dew loads images through Grain. A record is decoded, resized to `IMAGE_SIZE`, flipped and jittered for augmentation, and its caption is tokenized, all inside the worker processes. Every record draws its augmentation from its own random generator, so the same record is augmented the same way no matter how many workers or hosts produce the batches. The first run downloads the dataset through TFDS.

The loader returns factories rather than iterators, because a resumed run, or a run sharded across hosts, needs a fresh stream each time it asks for one.

In [ ]:
from dew.data.dataloaders import get_dataset_grain

data = get_dataset_grain("oxford_flowers102", batch_size=BATCH_SIZE, image_scale=IMAGE_SIZE,
                         worker_count=WORKER_COUNT, val_count=4 * BATCH_SIZE)
steps_per_epoch = data["train_len"] // BATCH_SIZE
print(f"{data['train_len']} training records, {data['val_len']} held out, "
      f"{steps_per_epoch} steps per epoch")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

batch = next(iter(data["val"]()))
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for ax, image in zip(axes.flat, batch["image"][:16]):
    ax.imshow(np.asarray(image))
    ax.axis("off")
plt.show()

## The schedule and the model

A diffusion model is three choices: the noise levels it trains on, the noise levels it walks when sampling, and what the network outputs. `get_diffusion_preset` hands back a matched set.

The `"edm"` preset is the formulation from *Elucidating the Design Space of Diffusion-Based Generative Models*. Training noise levels are drawn from a log-normal distribution, so the model spends its capacity where the denoising problem is hard. The network output is a scaled blend of the clean image and the noise, and the loss is weighted so every noise level contributes comparably. Sampling walks a Karras sigma schedule between `sigma_min` and `sigma_max`, spaced so the solver takes even steps in log space.

The model is a DiT built from a name and a config dict. At 64 pixels with a patch size of 4 it sees 256 patches per image. `dtype="bfloat16"` and `attention_impl="auto"` pick the fast kernels for your hardware; the parameter tree is identical either way, so a checkpoint trained with cuDNN attention on a GPU loads on a TPU.

In [ ]:
from dew.registry import apply_precision_policy, build_model
from dew.diffusion.transforms import get_diffusion_preset
from dew.inputs import DiffusionInputConfig

train_schedule, sample_schedule, transform = get_diffusion_preset("edm")

model = build_model("simple_dit", apply_precision_policy(
    "simple_dit", MODEL, dtype="bfloat16", attention_impl="auto"))

inputs = DiffusionInputConfig(
    sample_data_key="image",
    sample_data_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    conditions=[],  # unconditional: this model learns only what flowers look like
)

## The trainer

`ObjectiveTrainer` owns everything that is not the model or the data: it places the run on the device mesh, compiles one training step that covers the gradient, the update and the EMA, writes checkpoints and logs. The EMA is a slow copy of the weights that moves only a tenth of a percent toward the real weights on each step; sampling from the EMA weights is the standard trick that buys quality at no training cost.

At the end of every epoch the trainer runs a validation pass that generates four samples from the EMA weights with the Euler ancestral sampler, so you can watch the model learn while it trains. Set `WANDB_PROJECT` in the configuration cell to see them logged; without it the run prints to the terminal.

In [ ]:
import optax
from dew.sampling import EulerAncestralSampler
from dew.training import ObjectiveTrainer

wandb_config = None
if WANDB_PROJECT is not None:
    wandb_config = {
        "project": WANDB_PROJECT,
        "name": RUN_NAME,
        "config": {"model": MODEL, "image_size": IMAGE_SIZE, "batch_size": BATCH_SIZE,
                   "epochs": EPOCHS, "learning_rate": LEARNING_RATE},
    }

trainer = ObjectiveTrainer(
    model, optax.adamw(LEARNING_RATE),
    input_config=inputs,
    noise_schedule=train_schedule,
    model_output_transform=transform,
    rngs=jax.random.PRNGKey(SEED),
    name=RUN_NAME,
    wandb_config=wandb_config,
    log_every=50,
)

n_params = sum(p.size for p in jax.tree_util.tree_leaves(trainer.state.params))
print(f"{n_params / 1e6:.1f}M parameters")

In [ ]:
state = trainer.fit(
    data,
    training_steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    val_steps_per_epoch=1,
    sampler_class=EulerAncestralSampler,
    sampling_noise_schedule=sample_schedule,
)

## Sample from the model

Sampling is the reverse of the training corruption: start from pure Gaussian noise and walk it back to an image, one solver step at a time. The Euler ancestral sampler is the stochastic variant. After every deterministic step it adds back a calibrated fraction of noise, which lets it reach further in fewer steps than the plain ODE solvers, at the cost of a different sample each run.

The sampler takes the same schedule pair and input config the trainer used, and the EMA weights. The samples come back in `[-1, 1]`, so the grid helper rescales them before saving.

In [ ]:
import numpy as np
from PIL import Image
from dew.sampling import EulerAncestralSampler

def save_grid(images, path, cols=8):
    '''Save a grid of [-1, 1] images and display it inline.'''
    frames = np.clip((np.asarray(images) + 1) * 127.5, 0, 255).astype(np.uint8)
    rows = (len(frames) + cols - 1) // cols
    h, w, c = frames.shape[1:]
    grid = frames.reshape(rows, cols, h, w, c).transpose(0, 2, 1, 3, 4).reshape(rows * h, cols * w, c)
    import os
    os.makedirs(os.path.dirname(path), exist_ok=True)
    Image.fromarray(grid).save(path)
    plt.figure(figsize=(cols * 1.5, rows * 1.5))
    plt.imshow(grid)
    plt.axis("off")
    plt.show()
    return path

sampler = EulerAncestralSampler(model, sample_schedule, transform, inputs)
images = sampler.generate_samples(params=state.ema_params, num_samples=SAMPLES,
                                  resolution=IMAGE_SIZE, diffusion_steps=SAMPLE_STEPS)
save_grid(images, "samples/02-flowers-edm.png")

In [ ]:
import jax.numpy as jnp
from dew.sampling.loading import load_from_checkpoint

restored = load_from_checkpoint(f"checkpoints/{RUN_NAME}", step="best")
n_leaves = len(jax.tree_util.tree_leaves(restored.ema_params))
print(f"restored the EMA weights ({n_leaves} arrays) from step {int(restored.step)}")

restored_images = sampler.generate_samples(params=restored.ema_params, num_samples=4,
                                           resolution=IMAGE_SIZE, diffusion_steps=SAMPLE_STEPS)
save_grid(restored_images, "samples/02-flowers-edm-restored.png", cols=4)

## Logging the run

Set `WANDB_PROJECT` in the configuration cell and run the notebook again to log the run to [Weights & Biases](https://wandb.ai). Everything above stays the same; the trainer just reports instead of only printing. Per step: the loss and its throughput. Per epoch: the average loss, the epoch time, and the four validation samples as they improve. When the run ranks among the best in the project by loss, the newest checkpoint is pushed to the model registry, which is what the inference pipelines load from.

A logged run is resumable, comparable against other runs in the same project, and its config is stored with it, so any sample can be traced back to exactly the model and schedule that produced it.